# 02: Causal Feature Engineering & Cross-Vegetable Signals

### Notebook Purpose & Research Objectives
This notebook demonstrates the creation of the global feature matrix without future information leakage:
1. 8 causal price lags [1, 2, 3, 4, 8, 12, 26, 48].
2. Multi-scale causal rolling statistics [4, 8, 12, 26 weeks] (mean, std, min, max, skew).
3. Sinusoidal cyclical calendar embeddings (month and week-of-year).
4. Cross-vegetable category means and market-wide median percentage changes.
5. Causal missing-value indicators and forward filling (max 2 weeks).

---


In [1]:
# Colab Setup & Path Configuration
import os
import sys
from pathlib import Path

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    proj_dir = Path('/content/drive/MyDrive/veg_price_forecasting')
    if proj_dir.exists():
        os.chdir(proj_dir)
        sys.path.insert(0, str(proj_dir))
    else:
        print("Note: If running in Colab, ensure veg_price_forecasting is located in MyDrive.")
else:
    proj_dir = Path.cwd().resolve()
    if proj_dir.name == "notebooks":
        proj_dir = proj_dir.parent
    os.chdir(proj_dir)
    sys.path.insert(0, str(proj_dir))

print(f"Active Working Directory: {Path.cwd()}")


Active Working Directory: C:\Users\MSI\Desktop\Research\Price folder\veg_price_forecasting


### 1. Generate Global Features

In [2]:
from src.features import build_vegetable_features
from src.data import load_and_preprocess_vegetables
from src.utils import load_config

cfg = load_config("config.yaml")
df_wide, df_long = load_and_preprocess_vegetables(cfg)
df_feat = build_vegetable_features(df_long, cfg)
print(f"Generated feature matrix: {df_feat.shape} rows x {df_feat.shape[1]} columns")
print(f"Total engineered features: {len(df_feat.columns)}")


2026-10-01 15:25:51,705 [INFO] src.data: Raw data SHA-256 integrity check verified successfully.


2026-10-01 15:25:51,786 [INFO] src.data: Loaded 464 weeks and 14848 long records for 32 vegetables.


C:\Users\MSI\Desktop\Research\Price folder\veg_price_forecasting\src\features.py:50: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  market_stats["market_pct_change_lag1"] = market_stats["market_median_price"].pct_change(1)
C:\Users\MSI\Desktop\Research\Price folder\veg_price_forecasting\src\features.py:122: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  feat_df["pct_change_1w"] = imputed_price.pct_change(1).shift(1)
C:\Users\MSI\Desktop\Research\Price folder\veg_price_forecasting\src\features.py:123: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be remove

C:\Users\MSI\Desktop\Research\Price folder\veg_price_forecasting\src\features.py:122: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  feat_df["pct_change_1w"] = imputed_price.pct_change(1).shift(1)
C:\Users\MSI\Desktop\Research\Price folder\veg_price_forecasting\src\features.py:123: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  feat_df["pct_change_4w"] = imputed_price.pct_change(4).shift(1)
C:\Users\MSI\Desktop\Research\Price folder\veg_price_forecasting\src\features.py:124: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Eit

2026-10-01 15:25:52,476 [INFO] src.features: Engineered 71 total columns across 14848 observations (cross_veg=True).


Generated feature matrix: (14848, 71) rows x 71 columns
Total engineered features: 71


C:\Users\MSI\Desktop\Research\Price folder\veg_price_forecasting\src\features.py:122: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  feat_df["pct_change_1w"] = imputed_price.pct_change(1).shift(1)
C:\Users\MSI\Desktop\Research\Price folder\veg_price_forecasting\src\features.py:123: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  feat_df["pct_change_4w"] = imputed_price.pct_change(4).shift(1)
C:\Users\MSI\Desktop\Research\Price folder\veg_price_forecasting\src\features.py:124: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Eit

### 2. Inspect Engineered Feature Columns

In [3]:
lag_cols = [c for c in df_feat.columns if "lag_" in c]
roll_cols = [c for c in df_feat.columns if "roll_" in c]
cross_cols = [c for c in df_feat.columns if "cross_veg_" in c]

print(f"Lag features ({len(lag_cols)}): {lag_cols[:6]}...")
print(f"Rolling features ({len(roll_cols)}): {roll_cols[:6]}...")
print(f"Cross-vegetable features ({len(cross_cols)}): {cross_cols}")


Lag features (16): ['lag_1', 'log_lag_1', 'lag_2', 'log_lag_2', 'lag_3', 'log_lag_3']...
Rolling features (20): ['roll_mean_4', 'roll_std_4', 'roll_min_4', 'roll_max_4', 'roll_skew_4', 'roll_mean_8']...
Cross-vegetable features (0): []


### 3. Validate Causal Integrity (Zero Future Leakage)

In [4]:
sample_veg = df_feat[df_feat["product_name"] == "Carrot"].sort_values("week_index")
print("Carrot Feature Snapshot (Price vs Lag 1 vs Rolling Mean 4):")
print(sample_veg[["approx_date", "price_lkr", "lag_1", "roll_mean_4", "target_h1", "target_h4"]].head(8))


Carrot Feature Snapshot (Price vs Lag 1 vs Rolling Mean 4):
     approx_date  price_lkr   lag_1  roll_mean_4  target_h1  target_h4
4640  2017-01-04     139.71     NaN          NaN     132.41     125.56
4641  2017-01-11     132.41  139.71          NaN     130.26     131.20
4642  2017-01-18     130.26  132.41   136.060000     128.00     132.80
4643  2017-01-25     128.00  130.26   134.126667     125.56     129.30
4644  2017-02-04     125.56  128.00   132.595000     131.20     130.94
4645  2017-02-11     131.20  125.56   129.057500     132.80     130.56
4646  2017-02-18     132.80  131.20   128.755000     129.30     129.41
4647  2017-02-25     129.30  132.80   129.390000     130.94     136.26


---
### Research Findings & Plain Language Interpretation
The engineered feature matrix strictly obeys causality:
- Target values ($h=1, 4, 8, 12$) are strictly future-shifted relative to the forecast origin.
- All rolling statistics and lags depend solely on past observations.
- Cross-vegetable category means use lagged values to prevent intra-week simultaneous feedback leakage.
